In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression



# Helper Functions

In [ ]:
def fit_linear_function(linear_range, x, y):
    coeff = np.polyfit(linear_range[x], linear_range[y], 1)
    r2 = np.corrcoef(linear_range[x], linear_range[y])[0,1]**2
    return coeff, r2

def plot_linear_fit(data, linear_range, x, y, text_start_x, text_start_y, text_y_spacing, ax):
    coeff, r2 = fit_linear_function(linear_range, x, y)
    vals= np.polyval(coeff, data[x])
    ax.plot(data[x], vals, color='b')
    ax.text(text_start_x, text_start_y, 'Slope: %.4f' % (coeff[0]))
    ax.text(text_start_x, text_start_y - text_y_spacing, 'Intercept: %.4f' % (coeff[1]))
    ax.text(text_start_x, text_start_y - 2*text_y_spacing, 'R2: %.4f' % (r2))
    return coeff, r2

In [ ]:
def plot_nutrient_data(df, start_date, end_date, param, ax, check_valid = True, **kwargs):
    df.loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, label = param, **kwargs)


def plot_nutrient_data_by_bottle(df, start_date, end_date, param, ax, check_valid = True, **kwargs):
    first_rep = df['Bottle Replicate'] == 1
    print(first_rep)
    df[first_rep].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, label = 'First Bottle',  **kwargs)
    df[~first_rep].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', color = 'r', rot=45, label = 'Second Bottle',  **kwargs)


def plot_nutrient_data_by_rundate(df, start_date, end_date, param, ax, check_valid=True, **kwargs):
    rundates = df['AA500 Run Date'].unique()
    from itertools import cycle
    color_cycler = plt.rcParams['axes.prop_cycle'].by_key()['color']
    colors = cycle(color_cycler)
    for date, color in zip(rundates, colors):
        df[df['AA500 Run Date'] == date].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, color=color, label = 'Run on: ' + date.strftime('%m/%d/%Y'),  **kwargs)
    

def plot_nutrient_data_by_type(df, start_date, end_date, param, ax, check_valid=True, **kwargs):
    sampletypes = df['Sample Type'].unique()
    from itertools import cycle
    color_cycler = plt.rcParams['axes.prop_cycle'].by_key()['color']
    colors = cycle(color_cycler)
    for type, color in zip(sampletypes, colors):
        df[df['Sample Type'] == type].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, color=color, label = type,  **kwargs)

def plot_nutrient_data_by_volume(df, start_date, end_date, param, ax, check_valid=True, cmap='bwr', **kwargs):
    # Filter data by date range
    data = df.loc[start_date:end_date].reset_index()
    # Get water volume values for coloring
    volumes = data['Water Volume']
    scatter = ax.scatter(
        data['Sample Datetime'],
        data[param + ' mean'],
        c=volumes,
        cmap=cmap,
        label=param,
        **kwargs
    )
    ax.errorbar(
        data['Sample Datetime'],
        data[param + ' mean'],
        yerr=data[param + ' err'],
        fmt='none',
        ecolor='gray',
        alpha=0.5
    )
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Water Volume')
    ax.set_title(f'{param} by Water Volume')
    ax.set_xlabel('Sample Datetime')
    ax.set_ylabel(f'{param} mean')
    

# Import Data

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Processed Data/'
rme_results = pd.read_csv(result_dir+'rme_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)
rme_results = rme_results.drop(['2/15/25 3:30:00', '2-22-25 11:00']) # drop outliers
# Filter out VOL flagged results
rme_results_novol = rme_results[~rme_results['Nitrate QA'].str.contains('VOL')]
rme_results_vol = rme_results[rme_results['Nitrate QA'].str.contains('VOL')]

rme_results

In [ ]:
rme_cleaned = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv', index_col='Date/Time', parse_dates=True).dropna(subset=np.arange(220.0, 735.0, 2.5).astype(str))
rme_cleaned_uncorrected = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected.csv', index_col='Date/Time', parse_dates=True).dropna(subset=np.arange(220.0, 735.0, 2.5).astype(str))

rme_cleaned['second_derivative_no3_mgl_bias_correct'] = rme_cleaned['second_derivative_no3_mgl_correct']-.1138
rme_cleaned.loc[rme_cleaned['second_derivative_no3_mgl_bias_correct']<0, 'second_derivative_no3_mgl_bias_correct'] = 0

rme_cleaned_uncorrected['second_derivative_no3_mgl_bias_correct'] = rme_cleaned_uncorrected['second_derivative_no3_mgl_correct']-.1138
rme_cleaned_uncorrected.loc[rme_cleaned_uncorrected['second_derivative_no3_mgl_bias_correct']<0, 'second_derivative_no3_mgl_bias_correct'] = 0

scan_data = rme_cleaned[np.arange(220,735, 2.5).astype(str)]


In [ ]:
rme_cleaned_uncorrected

# PLSR

## Assign classes for stratification

In [ ]:
rme_results_novol.plot(y='Nitrate mean', kind='hist', bins=10, ylim=(0,25))

In [ ]:
def assign_class(df, val, thresholds, classes, new_col='Nitrate Class'):
    class_labels = pd.Series(index=df.index, dtype='object')
    vals = df[val]
    for threshold, c in zip(thresholds, classes):
        class_labels[(vals >= threshold[0]) & (vals < threshold[1])] = c
    df[new_col] = class_labels
    return df

rme_results_novol = assign_class(rme_results_novol, 'Nitrate mean', [(0, .1), (.1, .3), (.3, 10)], ['L', 'M', 'H'])
rme_results_novol.loc[rme_results_novol['Nitrate Class']=='H'].sort_values('Nitrate mean', ascending=False)


In [ ]:
nitrate_merged = pd.merge_asof(rme_results_novol, scan_data, left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('1h')).dropna(subset=np.arange(220.0, 735.0, 2.5).astype(str))
nitrate_merged

In [ ]:
nitrate_merged_uncorrected = pd.merge_asof(rme_results_novol, rme_cleaned_uncorrected[np.arange(220,735, 2.5).astype(str)], left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('1h')).dropna(subset=np.arange(220.0, 735.0, 2.5).astype(str))
nitrate_merged_uncorrected

## N_components selection

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.cross_decomposition import PLSRegression
import numpy as np

# Get indices for train/test split
indices = nitrate_merged.index
train_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=nitrate_merged['Nitrate Class'], random_state=42)

# Use indices to select rows in both datasets
X_train = nitrate_merged.loc[train_idx, np.arange(220.0, 735.0, 2.5).astype(str)]
X_test = nitrate_merged.loc[test_idx, np.arange(220.0, 735.0, 2.5).astype(str)]

X_train_uncorrected = nitrate_merged_uncorrected.loc[train_idx, np.arange(220.0, 735.0, 2.5).astype(str)]
X_test_uncorrected = nitrate_merged_uncorrected.loc[test_idx, np.arange(220.0, 735.0, 2.5).astype(str)]

y_train = nitrate_merged.loc[train_idx, 'Nitrate mean']
y_test = nitrate_merged.loc[test_idx, 'Nitrate mean']

# Pipeline (PLSR only, no scaling since scale=False in your workflow)
loo = LeaveOneOut()
n_components_range = range(1, 26)
correct_cv_scores = []

for n in n_components_range:
    pls = PLSRegression(n_components=n)
    pipe = Pipeline([('pls', pls)])
    scores = cross_val_score(pipe, X_train, y_train, cv=loo, scoring='neg_root_mean_squared_error')
    correct_cv_scores.append(np.mean(scores))

pos_correct_cv = np.multiply(correct_cv_scores, -1)

In [ ]:
correct_noscale_cv_scores = []

for n in n_components_range:
    pls = PLSRegression(n_components=n, scale=False)
    pipe = Pipeline([('pls', pls)])
    scores = cross_val_score(pipe, X_train, y_train, cv=loo, scoring='neg_root_mean_squared_error')
    correct_noscale_cv_scores.append(np.mean(scores))

pos_correct_noscale_cv = np.multiply(correct_noscale_cv_scores, -1)

In [ ]:
uncorrect_noscale_cv_scores = []

for n in n_components_range:
    pls = PLSRegression(n_components=n, scale=False)
    pipe = Pipeline([('pls', pls)])
    scores = cross_val_score(pipe, X_train_uncorrected, y_train, cv=loo, scoring='neg_root_mean_squared_error')
    uncorrect_noscale_cv_scores.append(np.mean(scores))

pos_uncorrect_noscale_cv = np.multiply(uncorrect_noscale_cv_scores, -1)


In [ ]:
uncorrect_scale_cv_scores = []

for n in n_components_range:
    pls = PLSRegression(n_components=n)
    pipe = Pipeline([('pls', pls)])
    scores = cross_val_score(pipe, X_train_uncorrected, y_train, cv=loo, scoring='neg_root_mean_squared_error')
    uncorrect_scale_cv_scores.append(np.mean(scores))

pos_uncorrect_scale_cv = np.multiply(uncorrect_scale_cv_scores, -1)


In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5), nrows=2, sharey=True)


ax[0].plot(n_components_range, pos_correct_cv, label='Corrected, Scaling=True')
ax[0].plot(n_components_range, pos_correct_noscale_cv, label='Corrected, Scaling=False')
ax[1].plot(n_components_range, pos_uncorrect_scale_cv, label='Uncorrected, Scaling=True')
ax[1].plot(n_components_range, pos_uncorrect_noscale_cv, label='Uncorrected, Scaling=False')
ax[0].set_title('Turbidity Corrected Cross Validation Results')
ax[1].set_title('Uncorrected Cross Validation Results')

for a in ax:
    a.set_ylabel('RMSECV')
    a.set_xlabel('N_Components')
    a.legend()

fig.tight_layout()

Observations:
- scaling seems to lead to worse results for turbidity corrected data, and the same results for uncorrected data. 
- seem to minimize RMSE at around 15 componenbts for all models
- I believe error uncorrected gives you less CV error, but lets check on the test set.

## Model fitting on entire train set, evaluation on test set

In [ ]:
test_results = pd.DataFrame()
test_results['Nitrate mean'] = y_test

scale_pls_correct = PLSRegression(n_components=5, scale=True)
noscale_pls_correct = PLSRegression(n_components=5, scale=False)
scale_pls_uncorrect = PLSRegression(n_components=5, scale=True)
noscale_pls_uncorrect = PLSRegression(n_components=5, scale=False)

scale_pls_correct.fit(X_train, y_train)
noscale_pls_correct.fit(X_train, y_train)
scale_pls_uncorrect.fit(X_train_uncorrected, y_train)
noscale_pls_uncorrect.fit(X_train_uncorrected, y_train)

test_results['Unscaled PLSR'] = scale_pls_correct.predict(X_test)
test_results['Scaled PLSR'] = noscale_pls_correct.predict(X_test)
test_results['Uncorrected Unscaled PLSR'] = scale_pls_uncorrect.predict(X_test)
test_results['Uncorrected Scaled PLSR'] = noscale_pls_uncorrect.predict(X_test)

fig, ax= plt.subplots(nrows = 2, ncols=2, figsize=(11,8.5), sharex=True, sharey=True)

test_results.plot(x='Nitrate mean', y='Scaled PLSR', kind='scatter', ax=ax[0,0], title='Scaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Scaled PLSR', .3, .2, .025, ax[0,0])

test_results.plot(x='Nitrate mean', y='Unscaled PLSR', kind='scatter', ax=ax[0,1], title='Unscaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Unscaled PLSR', .3, .2, .025, ax[0,1])

test_results.plot(x='Nitrate mean', y='Uncorrected Scaled PLSR', kind='scatter', ax=ax[1,0], title='Uncorrected Scaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Uncorrected Scaled PLSR', .3, .2, .025, ax[1,0])

test_results.plot(x='Nitrate mean', y='Uncorrected Unscaled PLSR', kind='scatter', ax=ax[1,1], title='Uncorrected Unscaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Uncorrected Unscaled PLSR', .3, .2, .025, ax[1,1])

for a in ax.flatten():
    a.set_ylim(-.05, .55)
    a.set_xlim(-.05,.55)

fig.suptitle('PLSR Results: N_components = 5')

fig.tight_layout()

In [ ]:
test_results = pd.DataFrame()
test_results['Nitrate mean'] = y_test

scale_pls_correct = PLSRegression(n_components=10, scale=True)
noscale_pls_correct = PLSRegression(n_components=10, scale=False)
scale_pls_uncorrect = PLSRegression(n_components=10, scale=True)
noscale_pls_uncorrect = PLSRegression(n_components=10, scale=False)

scale_pls_correct.fit(X_train, y_train)
noscale_pls_correct.fit(X_train, y_train)
scale_pls_uncorrect.fit(X_train_uncorrected, y_train)
noscale_pls_uncorrect.fit(X_train_uncorrected, y_train)

test_results['Unscaled PLSR'] = scale_pls_correct.predict(X_test)
test_results['Scaled PLSR'] = noscale_pls_correct.predict(X_test)
test_results['Uncorrected Unscaled PLSR'] = scale_pls_uncorrect.predict(X_test)
test_results['Uncorrected Scaled PLSR'] = noscale_pls_uncorrect.predict(X_test)

fig, ax= plt.subplots(nrows = 2, ncols=2, figsize=(11,8.5), sharex=True, sharey=True)

test_results.plot(x='Nitrate mean', y='Scaled PLSR', kind='scatter', ax=ax[0,0], title='Scaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Scaled PLSR', .3, .2, .025, ax[0,0])

test_results.plot(x='Nitrate mean', y='Unscaled PLSR', kind='scatter', ax=ax[0,1], title='Unscaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Unscaled PLSR', .3, .2, .025, ax[0,1])

test_results.plot(x='Nitrate mean', y='Uncorrected Scaled PLSR', kind='scatter', ax=ax[1,0], title='Uncorrected Scaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Uncorrected Scaled PLSR', .3, .2, .025, ax[1,0])

test_results.plot(x='Nitrate mean', y='Uncorrected Unscaled PLSR', kind='scatter', ax=ax[1,1], title='Uncorrected Unscaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Uncorrected Unscaled PLSR', .3, .2, .025, ax[1,1])

for a in ax.flatten():
    a.set_ylim(-.05, .55)
    a.set_xlim(-.05,.55)

fig.suptitle('PLSR Results: N_components = 10')

fig.tight_layout()

In [ ]:
test_results = pd.DataFrame()
test_results['Nitrate mean'] = y_test

scale_pls_correct = PLSRegression(n_components=15, scale=True)
noscale_pls_correct = PLSRegression(n_components=15, scale=False)
scale_pls_uncorrect = PLSRegression(n_components=15, scale=True)
noscale_pls_uncorrect = PLSRegression(n_components=15, scale=False)

scale_pls_correct.fit(X_train, y_train)
noscale_pls_correct.fit(X_train, y_train)
scale_pls_uncorrect.fit(X_train_uncorrected, y_train)
noscale_pls_uncorrect.fit(X_train_uncorrected, y_train)

test_results['Unscaled PLSR'] = scale_pls_correct.predict(X_test)
test_results['Scaled PLSR'] = noscale_pls_correct.predict(X_test)
test_results['Uncorrected Unscaled PLSR'] = scale_pls_uncorrect.predict(X_test)
test_results['Uncorrected Scaled PLSR'] = noscale_pls_uncorrect.predict(X_test)

fig, ax= plt.subplots(nrows = 2, ncols=2, figsize=(11,8.5), sharex=True, sharey=True)

test_results.plot(x='Nitrate mean', y='Scaled PLSR', kind='scatter', ax=ax[0,0], title='Scaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Scaled PLSR', .3, .2, .025, ax[0,0])

test_results.plot(x='Nitrate mean', y='Unscaled PLSR', kind='scatter', ax=ax[0,1], title='Unscaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Unscaled PLSR', .3, .2, .025, ax[0,1])

test_results.plot(x='Nitrate mean', y='Uncorrected Scaled PLSR', kind='scatter', ax=ax[1,0], title='Uncorrected Scaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Uncorrected Scaled PLSR', .3, .2, .025, ax[1,0])

test_results.plot(x='Nitrate mean', y='Uncorrected Unscaled PLSR', kind='scatter', ax=ax[1,1], title='Uncorrected Unscaled PLSR')
plot_linear_fit(test_results, test_results, 'Nitrate mean', 'Uncorrected Unscaled PLSR', .3, .2, .025, ax[1,1])

for a in ax.flatten():
    a.set_ylim(-.05, .55)
    a.set_xlim(-.05,.55)

fig.suptitle('PLSR Results: N_components = 15')

fig.tight_layout()

## Calculating for entire dataset

In [ ]:
scan_cols = np.arange(220,735, 2.5).astype(str)

scale_pls_correct = PLSRegression(n_components=10, scale=True)
noscale_pls_uncorrect = PLSRegression(n_components=5, scale=False)
scale_pls_correct.fit(X_train, y_train)
noscale_pls_uncorrect.fit(X_train_uncorrected, y_train)

rme_cleaned['Scaled PLSR'] = scale_pls_correct.predict(rme_cleaned[scan_cols])
rme_cleaned_uncorrected['Uncorrected Unscaled PLSR'] = noscale_pls_uncorrect.predict(rme_cleaned_uncorrected[scan_cols])
rme_cleaned['Uncorrected Unscaled PLSR'] = rme_cleaned_uncorrected['Uncorrected Unscaled PLSR']

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=5, sharex=True)

rme_cleaned.loc['02-01-25':'12-01-25'].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct')
rme_cleaned.loc['02-01-25':'12-01-25'].plot(y= 'second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title= 'second_derivative_no3_mgl_correct')
rme_cleaned.loc['02-01-25':'12-01-25'].plot(y= 'second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title= 'second_derivative_no3_mgl_bias_correct')
rme_cleaned.loc['02-01-25':'12-01-25'].plot(y= 'Scaled PLSR', ax=ax[3], rot=45, title= 'Scaled PLSR')
rme_cleaned_uncorrected.loc['2-01-25':'12-01-2025'].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR')

for a in ax:
    plot_nutrient_data(rme_results_novol, '02/01/25', '12/01/25', 'Nitrate', a, color='k')

fig.tight_layout()

## What's the deal with the June Spikes?

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '06-01-2025'
end_date = '07-01-2025'

rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title='second_derivative_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='second_derivative_no3_mgl_bias_correct')
rme_cleaned.loc[start_date:end_date].plot(y='Scaled PLSR', ax=ax[3], rot=45, title='Scaled PLSR')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[6], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')



for i in range(0,6):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '06-14-2025 14:00'
end_date = '06-14-2025 20:00'

rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title='second_derivative_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='second_derivative_no3_mgl_bias_correct')
rme_cleaned.loc[start_date:end_date].plot(y='Scaled PLSR', ax=ax[3], rot=45, title='Scaled PLSR')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[6], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')



for i in range(0,6):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '06-17-2025'
end_date = '06-21-2025'

rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title='second_derivative_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='second_derivative_no3_mgl_bias_correct')
rme_cleaned.loc[start_date:end_date].plot(y='Scaled PLSR', ax=ax[3], rot=45, title='Scaled PLSR')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[6], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')



for i in range(0,6):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')

fig.tight_layout()

spikes in calculated nitrate correspond to spikes in absornance

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '06-19-2025 14:00'
end_date = '06-19-2025 17:00'

rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title='second_derivative_no3_mgl_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='second_derivative_no3_mgl_bias_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='Scaled PLSR', ax=ax[3], rot=45, title='Scaled PLSR', marker='*')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Corrected Absorbance', marker='*')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[6], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance', marker='*')



for i in range(0,6):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '06-08-2025 16:00'
end_date = '06-08-2025 20:00'

rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title='second_derivative_no3_mgl_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='second_derivative_no3_mgl_bias_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='Scaled PLSR', ax=ax[3], rot=45, title='Scaled PLSR', marker='*')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Corrected Absorbance', marker='*')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[6], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance', marker='*')



for i in range(0,6):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')

fig.tight_layout()

its not just one datapoint - its a spike up to a really high absorbance, that decays back to noirmal in an hour or two

### What if we try looking at spectra from these weird periods?

In [ ]:
start_date = '06-19-2025 14:30'
end_date = '06-19-2025 15:45'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 735, 2.5).astype(str)].T
problem_spectra.plot()

hmm, one feature of problematic spectra are that 220 is not the maximum absorbance

In [ ]:
start_date = '06-08-2025 16:15'
end_date = '06-08-2025 17:15'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 735, 2.5).astype(str)].T
problem_spectra.plot()

In [ ]:
start_date = '06-08-2025 18:30'
end_date = '06-08-2025 19:45'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 735, 2.5).astype(str)].T
problem_spectra.plot()

hmm, don't see that flattening at 220 in this one. what other are weird?
- flat spots around 270 and 320
- derivative from 232.5 to 257.5

In [ ]:
start_date = '06-08-2025 18:30'
end_date = '06-08-2025 19:45'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 377.5, 2.5).astype(str)].T
problem_spectra.plot()

In [ ]:
start_date = '06-08-2025 18:30'
end_date = '06-08-2025 19:45'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 272.5, 2.5).astype(str)].T
problem_spectra.plot()

In [ ]:
start_date = '06-19-2025 14:30'
end_date = '06-19-2025 15:45'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 272.5, 2.5).astype(str)].T
problem_spectra.plot()

### what about the derivative at a certain threshold?

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '02-24-2025 02:00'
end_date = '02-24-2025 04:00'

rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title='second_derivative_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='second_derivative_no3_mgl_bias_correct')
rme_cleaned.loc[start_date:end_date].plot(y='Scaled PLSR', ax=ax[3], rot=45, title='Scaled PLSR')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax[6], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')



for i in range(0,5):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(11,6), sharey=True)

start_date = '06-19-2025 14:30'
end_date = '06-19-2025 15:45'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 272.5, 2.5).astype(str)].T
problem_spectra.plot(ax=ax)

start_date = '02-24-2025 02:00'
end_date = '02-24-2025 04:00'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 272.5, 2.5).astype(str)].T
problem_spectra.plot(ax=ax)




In [ ]:
fig, ax = plt.subplots(figsize=(11,6), sharey=True)

start_date = '06-08-2025 18:30'
end_date = '06-08-2025 19:45'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 272.5, 2.5).astype(str)].T
problem_spectra.plot(ax=ax)

start_date = '02-24-2025 02:00'
end_date = '02-24-2025 04:00'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 272.5, 2.5).astype(str)].T
problem_spectra.plot(ax=ax)


In [ ]:
fig, ax = plt.subplots(figsize=(11,6), sharey=True)

start_date = '06-08-2025 16:15'
end_date = '06-08-2025 17:15'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 272.5, 2.5).astype(str)].T
problem_spectra.plot(ax=ax)

start_date = '02-24-2025 02:00'
end_date = '02-24-2025 04:00'

problem_spectra= rme_cleaned.loc[start_date:end_date, np.arange(220.0, 272.5, 2.5).astype(str)].T
problem_spectra.plot(ax=ax)


- first derivative between 245 and 257.5 looks promising - in high nitrate peak on 2-24, its pretty flat there, but pretty slopy in weird peaks

In [ ]:
rme_scan_smooth = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected_smooth.csv', index_col='Date/Time', parse_dates=True).dropna(subset=np.arange(220.0, 735.0, 2.5).astype(str))
rme_scan_1d = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected_1d.csv', index_col='Date/Time', parse_dates=True).dropna(subset=np.arange(220.0, 735.0, 2.5).astype(str))
rme_scan_2d = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected_2d.csv', index_col='Date/Time', parse_dates=True).dropna(subset=np.arange(220.0, 735.0, 2.5).astype(str))


In [ ]:
fig, ax = plt.subplots(figsize=(15, 30), nrows=7, sharey=False, sharex=True)

start_date = '06-08-2025 14:00'
end_date = '06-08-2025 20:00'

rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title='second_derivative_no3_mgl_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='second_derivative_no3_mgl_bias_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='Scaled PLSR', ax=ax[3], rot=45, title='Scaled PLSR', marker='*')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', marker='*')
rme_scan_1d.loc[start_date:end_date].plot(y=np.arange(245.0, 255, 2.5).astype(str), ax=ax[5], rot=45, title='1st Derivative of Spectrum', marker='*')
rme_scan_2d.loc[start_date:end_date].plot(y=np.arange(245.0, 255, 2.5).astype(str), ax=ax[6], rot=45, title='2nd Derivative of Spectrum', marker='*')
rme_scan_2d.loc[start_date:end_date].plot(y=np.arange(220.0, 237.5, 2.5).astype(str), ax=ax[6], rot=45, title='2nd Derivative of Spectrum', marker='*')

#ax[5].set_ylim(-4, 0)
ax[5].axhline(-.6, color='r', linestyle='--')
ax[6].axhline(.16, color='r', linestyle='--')

for i in range(0,6):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 30), nrows=7, sharey=False, sharex=True)

start_date = '06-19-25 14:00'
end_date = '06-19-25 17:00'

rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title='second_derivative_no3_mgl_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='second_derivative_no3_mgl_bias_correct', marker='*')
rme_cleaned.loc[start_date:end_date].plot(y='Scaled PLSR', ax=ax[3], rot=45, title='Scaled PLSR', marker='*')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', marker='*')
rme_scan_1d.loc[start_date:end_date].plot(y=np.arange(245.0, 255, 2.5).astype(str), ax=ax[5], rot=45, title='1st Derivative of Spectrum', marker='*')
rme_scan_2d.loc[start_date:end_date].plot(y=np.arange(245.0, 255, 2.5).astype(str), ax=ax[6], rot=45, title='2nd Derivative of Spectrum', marker='*')
rme_scan_2d.loc[start_date:end_date].plot(y=np.arange(220.0, 237.5, 2.5).astype(str), ax=ax[6], rot=45, title='2nd Derivative of Spectrum', marker='*')

#ax[5].set_ylim(-4, 0)
ax[5].axhline(-.6, color='r', linestyle='--')
ax[6].axhline(.16, color='r', linestyle='--')

for i in range(0,6):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 30), nrows=7, sharey=False, sharex=True)

start_date = '2-24-25 00:00'
end_date = '2-24-25 12:00'

rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title='second_derivative_no3_mgl_correct')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='second_derivative_no3_mgl_bias_correct')
rme_cleaned.loc[start_date:end_date].plot(y='Scaled PLSR', ax=ax[3], rot=45, title='Scaled PLSR')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR')
rme_scan_1d.loc[start_date:end_date].plot(y=np.arange(245, 255, 2.5).astype(str), ax=ax[5], rot=45, title='1st Derivative of Spectrum')
rme_scan_2d.loc[start_date:end_date].plot(y=np.arange(245.0, 255, 2.5).astype(str), ax=ax[6], rot=45, title='2nd Derivative of Spectrum', marker='*')
rme_scan_2d.loc[start_date:end_date].plot(y=np.arange(220.0, 237.5, 2.5).astype(str), ax=ax[6], rot=45, title='2nd Derivative of Spectrum', marker='*')

#ax[5].set_ylim(-4, 0)
ax[5].axhline(-.65, color='r', linestyle='--')
ax[6].axhline(.2, color='r', linestyle='--')

for i in range(0,5):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')

fig.tight_layout()

-0.65  seems like a really good threshold for 1st derivative at 247.5 nm. lets try it

In [ ]:
bool_idx = rme_scan_1d['247.5'] < -.65
bool_idx.sum()/bool_idx.count()

In [ ]:
bool_idx = rme_scan_2d['247.5'] >.2
bool_idx.sum()/bool_idx.count()

wow 1st derivative threshold thats actually 10% of readings, while second derivative is only 1%

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '02-01-2025'
end_date = '11-01-2025'

rme_cleaned_uncorrected[rme_scan_1d['247.5'] >= -.5].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[0], rot=45, title='Threshold = -.5')
rme_cleaned_uncorrected[rme_scan_1d['247.5'] >= -.6].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[1], rot=45, title='Threshold = -.6')
rme_cleaned_uncorrected[rme_scan_1d['247.5'] >= -.7].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[2], rot=45, title='Threshold = -.7')
rme_cleaned_uncorrected[rme_scan_1d['247.5'] >= -.8].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[3], rot=45, title='Threshold = -.8')
rme_cleaned_uncorrected[rme_scan_1d['247.5'] >= -.9].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Threshold = -.9')
rme_cleaned_uncorrected[rme_scan_1d['247.5'] >= -1].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[5], rot=45, title='Threshold = -1')
rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[6], label='Unfiltered', rot=45, title='Unfiltered')




for i in range(0,7):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')
    ax[i].set_ylim(-.1, 0.6)

fig.suptitle('Uncorected Unscaled PLSR Filtered by First Derivative')
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '02-01-2025'
end_date = '11-01-2025'

rme_cleaned[rme_scan_1d['247.5'] >= -.5].loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='Threshold = -.5')
rme_cleaned[rme_scan_1d['247.5'] >= -.6].loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[1], rot=45, title='Threshold = -.6')
rme_cleaned[rme_scan_1d['247.5'] >= -.7].loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[2], rot=45, title='Threshold = -.7')
rme_cleaned[rme_scan_1d['247.5'] >= -.8].loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[3], rot=45, title='Threshold = -.8')
rme_cleaned[rme_scan_1d['247.5'] >= -.9].loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[4], rot=45, title='Threshold = -.9')
rme_cleaned[rme_scan_1d['247.5'] >= -1].loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[5], rot=45, title='Threshold = -1')
rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[6], label='Unfiltered', rot=45, title='Unfiltered')




for i in range(0,7):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')
    ax[i].set_ylim(-.1, 0.6)


fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '02-01-2025'
end_date = '11-01-2025'

rme_cleaned_uncorrected[rme_scan_1d['247.5'] >= -.5].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[0], rot=45, title='Threshold = -.5')
rme_cleaned[rme_scan_1d['247.5'] >= -.6].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[1], rot=45, title='Threshold = -.6')
rme_cleaned[rme_scan_1d['247.5'] >= -.7].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='Threshold = -.7')
rme_cleaned[rme_scan_1d['247.5'] >= -.8].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[3], rot=45, title='Threshold = -.8')
rme_cleaned[rme_scan_1d['247.5'] >= -.9].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[4], rot=45, title='Threshold = -.9')
rme_cleaned[rme_scan_1d['247.5'] >= -1].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[5], rot=45, title='Threshold = -1')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[6], label='Unfiltered', rot=45, title='Unfiltered')




for i in range(0,7):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')
    ax[i].set_ylim(-.1, 0.6)


fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '07-01-2025'
end_date = '09-01-2025'

rme_cleaned_uncorrected[(rme_scan_2d['247.5'] <= .12) & (rme_scan_1d['247.5'] >= -.5)].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[0], rot=45, title='Threshold = .12')
rme_cleaned_uncorrected[(rme_scan_2d['247.5'] <= .13) & (rme_scan_1d['247.5'] >= -.6)].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[1], rot=45, title='Threshold = .13')
rme_cleaned_uncorrected[(rme_scan_2d['247.5'] <= .14) & (rme_scan_1d['247.5'] >= -.7)].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[2], rot=45, title='Threshold = .14')
rme_cleaned_uncorrected[(rme_scan_2d['247.5'] <= .15) & (rme_scan_1d['247.5'] >= -.8)].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[3], rot=45, title='Threshold = .15')
rme_cleaned_uncorrected[(rme_scan_2d['247.5'] <= .16) & (rme_scan_1d['247.5'] >= -.9)].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Threshold = .16')
rme_cleaned_uncorrected[(rme_scan_2d['247.5'] <= .17) & (rme_scan_1d['247.5'] >= -1)].loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[5], rot=45, title='Threshold = .17')
rme_cleaned.loc[start_date:end_date].plot(y='two_wavelength_no3_mgl_correct', ax=ax[6], label='Unfiltered', rot=45, title='Unfiltered')




for i in range(0,7):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')
    ax[i].set_ylim(-.1, 0.6)
fig.suptitle('Uncorrected Unscaled PLSR Filtered by 2nd Derivative')
fig.tight_layout()

In [ ]:
start_date = '02-01-2025'
end_date = '11-01-2025'

fig1, ax1 = plt.subplots(nrows=3, figsize=(15,15), sharey=False, sharex=True)

rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax1[0], rot=45, title='Unfiltered')
rme_cleaned_uncorrected.loc[(rme_scan_2d['227.5'] >= .13) & (rme_scan_2d['227.5'] <= 1.4)].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax1[1], rot=45, title='.15 < 227.5 2d Threshold < 1.3')
rme_scan_2d.plot(y='227.5', figsize=(11,5), ylim= (0, 1.5), ax=ax1[2])
ax1[2].axhline(.15, color='r', linestyle='--')

for i in range(0,2):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax1[i], color='k')
    ax1[i].set_ylim(-.1, 0.6)
fig1.suptitle('Second Derivative Calibration Filtered by 2nd Derivative')
fig1.tight_layout()
ax1[2].axhline(1.4, color='r', linestyle='--')



In [ ]:
start_date = '02-01-2025'
end_date = '11-01-2025'

fig1, ax1 = plt.subplots(nrows=4, figsize=(15,30), sharey=False, sharex=True)

rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax1[0], rot=45, title='Unfiltered')
rme_cleaned_uncorrected.loc[(rme_scan_2d['227.5'] >= .13) & (rme_scan_2d['227.5'] <= 1.4)].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax1[1], rot=45, title='.15 < 227.5 2d Threshold < 1.3')
rme_scan_2d.loc[start_date:end_date].plot(y='227.5', figsize=(11,5), ylim= (0, 1.5), ax=ax1[2], title='2nd Derivative at 227.5nm')
ax1[2].axhline(.15, color='r', linestyle='--')
ax1[3].axhline(55, color='r', linestyle='--')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=['220.0', '265.0'], ax=ax1[3], rot=45, title='Absorbance')


for i in range(0,2):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax1[i], color='k')
    ax1[i].set_ylim(-.1, 0.6)
fig1.suptitle('Second Derivative Calibration Filtered by 2nd Derivative')
fig1.tight_layout()
ax1[2].axhline(1.4, color='r', linestyle='--')



In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), nrows=7, sharey=False, sharex=True)

start_date = '02-01-2025'
end_date = '11-01-2025'

rme_cleaned_uncorrected[rme_scan_1d['247.5'] >= -.5].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[0], rot=45, title='Threshold = -.5')
rme_cleaned[rme_scan_1d['247.5'] >= -.6].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[1], rot=45, title='Threshold = -.6')
rme_cleaned[rme_scan_1d['247.5'] >= -.7].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title='Threshold = -.7')
rme_cleaned[rme_scan_1d['247.5'] >= -.8].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[3], rot=45, title='Threshold = -.8')
rme_cleaned[rme_scan_1d['247.5'] >= -.9].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[4], rot=45, title='Threshold = -.9')
rme_cleaned[rme_scan_1d['247.5'] >= -1].loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[5], rot=45, title='Threshold = -1')
rme_cleaned.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[6], label='Unfiltered', rot=45, title='Unfiltered')




for i in range(0,7):
    plot_nutrient_data(rme_results_novol, start_date, end_date, 'Nitrate', ax[i], color='k')
    ax[i].set_ylim(-.1, 0.6)


fig.tight_layout()

### What if instead of filtering on an absorbance of over 55 (which can be valid for nitrate in some cases) what about other wavelengths?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5), nrows=2, sharey=False, sharex=True)

start_date = '06-19-2025 14:00'
end_date = '06-19-2025 17:00'

rme_cleaned.loc[start_date:end_date].plot(y=np.arange(220.0, 500, 20).astype(str), ax=ax[0], rot=45, title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=np.arange(220.0, 500, 20).astype(str), ax=ax[1], rot=45, title = 'Uncorrected Absorbance')

ax[0].set_ylim(-1,10)

ax[1].set_ylim(-1,10)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5), nrows=3, sharey=False, sharex=True)

start_date = '06-19-2025 14:00'
end_date = '06-19-2025 17:00'

rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[0], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=np.arange(400.0, 420.00, 2.5).astype(str), ax=ax[1], rot=45, title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=np.arange(400.0, 420.00, 2.5).astype(str), ax=ax[2], rot=45, title = 'Uncorrected Absorbance')

ax[1].axhline(0, color='r', linestyle='--')
ax[1].axhline(-.1, color='r', linestyle='--')

fig.tight_layout()

can we use 400nm corrected being negative?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5), nrows=3, sharey=False, sharex=True)

start_date = '06-08-2025 16:00'
end_date = '06-08-2025 20:00'

rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[0], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=np.arange(400.0, 420.00, 2.5).astype(str), ax=ax[1], rot=45, title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=np.arange(400.0, 420.00, 2.5).astype(str), ax=ax[2], rot=45, title = 'Uncorrected Absorbance')

ax[1].axhline(0, color='r', linestyle='--')
ax[1].axhline(-.1, color='r', linestyle='--')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5), nrows=3, sharey=False, sharex=True)

start_date = '06-14-2025 14:00'
end_date = '06-14-2025 20:00'

rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[0], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=np.arange(400.0, 420.00, 2.5).astype(str), ax=ax[1], rot=45, title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=np.arange(400.0, 420.00, 2.5).astype(str), ax=ax[2], rot=45, title = 'Uncorrected Absorbance')

ax[1].axhline(0, color='r', linestyle='--')
ax[1].axhline(-.1, color='r', linestyle='--')

fig.tight_layout()

actiually 410 is looking a little better

## Trying 410nm

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=5, sharex=True)

rme_cleaned_410 = rme_cleaned[rme_cleaned['410.0']>0]
rme_cleaned_uncorrected_410 = rme_cleaned_uncorrected[rme_cleaned['410.0']>0]

rme_cleaned_410.loc['02-01-25':'02-01-26'].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct')
rme_cleaned_410.loc['02-01-25':'02-01-26'].plot(y= 'second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title= 'second_derivative_no3_mgl_correct')
rme_cleaned_410.loc['02-01-25':'02-01-26'].plot(y= 'second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title= 'second_derivative_no3_mgl_bias_correct')
rme_cleaned_410.loc['02-01-25':'02-01-26'].plot(y= 'Scaled PLSR', ax=ax[3], rot=45, title= 'Scaled PLSR')
rme_cleaned_uncorrected_410.loc['2-01-25':'02-01-2026'].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR')

for a in ax:
    plot_nutrient_data(rme_results_novol, '02/01/26', '02/01/26', 'Nitrate', a, color='k')

fig.tight_layout()

- truncates that nice snowmelt peak we get

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=5, sharex=True)

rme_cleaned_410 = rme_cleaned[rme_cleaned['410.0']>0]
rme_cleaned_uncorrected_410 = rme_cleaned_uncorrected[rme_cleaned['410.0']>0]

rme_cleaned_410.loc['07-01-25':'11-01-25'].plot(y='two_wavelength_no3_mgl_correct', ax=ax[0], rot=45, title='two_wavelength_no3_mgl_correct')
rme_cleaned_410.loc['07-01-25':'11-01-25'].plot(y= 'second_derivative_no3_mgl_correct', ax=ax[1], rot=45, title= 'second_derivative_no3_mgl_correct')
rme_cleaned_410.loc['07-01-25':'11-01-25'].plot(y= 'second_derivative_no3_mgl_bias_correct', ax=ax[2], rot=45, title= 'second_derivative_no3_mgl_bias_correct')
rme_cleaned_410.loc['07-01-25':'11-01-25'].plot(y= 'Scaled PLSR', ax=ax[3], rot=45, title= 'Scaled PLSR')
rme_cleaned_uncorrected_410.loc['7-01-25':'11-01-2025'].plot(y='Uncorrected Unscaled PLSR', ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR')

for a in ax:
    plot_nutrient_data(rme_results_novol, '07/01/25', '11/01/25', 'Nitrate', a, color='k')

fig.tight_layout()

doesn't solve problems with july on 

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5), nrows=3, sharey=False, sharex=True)

start_date = '06-08-2025 16:00'
end_date = '06-08-2025 20:00'

rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[0], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=np.arange(400.0, 420.00, 2.5).astype(str), ax=ax[1], rot=45, title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=np.arange(400.0, 420.00, 2.5).astype(str), ax=ax[2], rot=45, title = 'Uncorrected Absorbance')

ax[1].axhline(0, color='r', linestyle='--')
ax[1].axhline(-.1, color='r', linestyle='--')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5), nrows=3, sharey=False, sharex=True)

start_date = '07-01-2025'
end_date = '08-01-2025'

rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[0], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=np.arange(400, 500.0, 20).astype(str), ax=ax[1], rot=45, title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=np.arange(400.0, 500.0, 20).astype(str), ax=ax[2], rot=45, title = 'Uncorrected Absorbance')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5), nrows=3, sharey=False, sharex=True)

start_date = '09-01-2025'
end_date = '10-01-2025'

rme_cleaned_uncorrected.loc[start_date:end_date].plot(y='Uncorrected Unscaled PLSR', ax=ax[0], rot=45, title='Uncorrected Unscaled PLSR')
rme_cleaned.loc[start_date:end_date].plot(y=np.arange(300.0, 500.0, 20).astype(str), ax=ax[1], rot=45, title = 'Corrected Absorbance')
rme_cleaned_uncorrected.loc[start_date:end_date].plot(y=np.arange(300.0, 500.0, 20).astype(str), ax=ax[2], rot=45, title = 'Uncorrected Absorbance')

fig.tight_layout()

# PCA

## Corrected

In [ ]:
scaler = StandardScaler()
scan_data_scaled = scaler.fit_transform(scan_data)

# Fit PCA
pca = PCA()
principal_components = pca.fit_transform(scan_data_scaled)
pca_results = pd.DataFrame()
pca_results['Explained Variance'] = pca.explained_variance_ratio_
pca_results['Cumulative Explained Variance'] = pca_results['Explained Variance'].cumsum()
pca_results.index = pca_results.index + 1

fig, ax = plt.subplots()
pca_results.plot(y='Cumulative Explained Variance', ylabel = 'Cumulative Explained Variance', xlabel = 'Principal Components', xlim=(0,10), ax=ax)
ax.axhline(y=.95, color = 'r', linestyle = '--')


- 4 components explains 95% of variance

In [ ]:
# Create a DataFrame for the principal components

pca = PCA(n_components=2)
principal_components = pca.fit_transform(scan_data_scaled)
pc_df = pd.DataFrame(principal_components, index=scan_data.index, columns=['PC1', 'PC2'])


# Plot the first two principal components
plt.figure(figsize=(10,6))
plt.scatter(pc_df['PC1'], pc_df['PC2'], alpha=0.7)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of SCAN Data')
plt.grid(True)
plt.show()

# Explained variance
print("Explained variance ratio:", pca.explained_variance_ratio_)

## Uncorrected

In [ ]:
scaler = StandardScaler()
scan_data_scaled = scaler.fit_transform(rme_cleaned_uncorrected[np.arange(220,735, 2.5).astype(str)])

# Fit PCA
pca = PCA()
principal_components = pca.fit_transform(scan_data_scaled)
pca_results = pd.DataFrame()
pca_results['Explained Variance'] = pca.explained_variance_ratio_
pca_results['Cumulative Explained Variance'] = pca_results['Explained Variance'].cumsum()
pca_results.index = pca_results.index + 1

fig, ax = plt.subplots()
pca_results.plot(y='Cumulative Explained Variance', ylabel = 'Cumulative Explained Variance', xlabel = 'Principal Components', xlim=(0,5), ax=ax)
ax.axhline(y=.95, color = 'r', linestyle = '--')


- 2 components explains 95% of variance for uncorrected S::CAN data

In [ ]:
# Create a DataFrame for the principal components

pca = PCA(n_components=2)
principal_components = pca.fit_transform(scan_data_scaled)
pc_df = pd.DataFrame(principal_components, index=scan_data.index, columns=['PC1', 'PC2'])


# Plot the first two principal components
plt.figure(figsize=(10,6))
plt.scatter(pc_df['PC1'], pc_df['PC2'], alpha=0.7)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of SCAN Data')
plt.grid(True)
plt.show()

# Explained variance
print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
scan_data_scaled

In [ ]:
X = nitrate_merged_uncorrected[np.arange(220.0, 735.0, 2.5).astype(str)]
Y = nitrate_merged_uncorrected['Nitrate mean']

pls4 = PLSRegression(n_components=4, scale=False)
pls4.fit(X, Y)

Y_pred = pls4.predict(rme_cleaned_uncorrected[np.arange(220.0, 735.0, 2.5).astype(str)])
rme_cleaned['PLSR Uncorrected'] = Y_pred

# What happened in December?

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=7, sharex=True)
plot_nutrient_data(rme_results, '11/01/25', '02/01/26', 'Nitrate', ax[0], color='k')
rme_cleaned.loc['11-01-25':'02-01-26'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

plot_nutrient_data(rme_results, '11/01/25', '02/01/26', 'Nitrate', ax[1], color='k')
rme_cleaned.loc['11-01-25':'02-01-26'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')
#rme_cleaned_uncorrected.loc['11-01-25':'02-01-26'].plot(y=['second_derivative'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:orange', secondary_y=True)


plot_nutrient_data(rme_results, '11/01/25', '02/01/26', 'Nitrate', ax[2], color='k')
rme_cleaned.loc['11-01-25':'02-01-26'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


plot_nutrient_data(rme_results, '11/01/25', '02/01/26', 'Nitrate', ax[3], color='k')
rme_cleaned.loc['11-01-25':'02-01-26'].plot(y=['Scaled PLSR'], ax=ax[3], rot=45, title='Scaled PLSR', c='tab:blue')

plot_nutrient_data(rme_results, '11/01/25', '02/01/26', 'Nitrate', ax[4], color='k')
rme_cleaned_uncorrected.loc['11-01-25':'02-01-26'].plot(y=['Uncorrected Unscaled PLSR'], ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', c='tab:blue')

rme_cleaned_uncorrected.loc['11-01-25':'02-01-26'].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')
rme_cleaned.loc['11-01-25':'02-01-26'].plot(y=['220.0'], ax=ax[6], rot=45, label=['220nm Absorbance'], title='Corrected Absorbance')
rme_cleaned.loc['11-01-25':'02-01-26'].plot(y=['265.0'], ax=ax[6], rot=45, label=['265nm Absorbance'], title='Corrected Absorbance')

fig.tight_layout()

# Plotting with S-CAN Data

In [ ]:
fig, ax= plt.subplots(figsize=(15, 9), nrows=2, sharex=True)
plot_nutrient_data_by_bottle(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[0])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct', 'Scaled PLSR'], color=['g', 'y','b','c'], ax=ax[0], rot=45, title='All Data')

plot_nutrient_data_by_bottle(rme_results[rme_results['Nitrate QA']==''], '02/01/25', '11/01/25', 'Nitrate', ax[1])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct', 'Scaled PLSR'], color=['g', 'y','b','c'], ax=ax[1], rot=45, title='Excluding Flagged Data')

fig.tight_layout()

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=7, sharex=True)
plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[0], color='k')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[1], color='k')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')
#rme_cleaned_uncorrected.loc['02-01-25':'11-01-25'].plot(y=['second_derivative'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:orange', secondary_y=True)


plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[2], color='k')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[3], color='k')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['Scaled PLSR'], ax=ax[3], rot=45, title='Scaled PLSR', c='tab:blue')

plot_nutrient_data(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[4], color='k')
rme_cleaned_uncorrected.loc['02-01-25':'11-01-25'].plot(y=['Uncorrected Unscaled PLSR'], ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', c='tab:blue')

rme_cleaned_uncorrected.loc['02-01-25':'11-01-25'].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['220.0'], ax=ax[6], rot=45, label=['220nm Absorbance'], title='Corrected Absorbance')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['265.0'], ax=ax[6], rot=45, label=['265nm Absorbance'], title='Corrected Absorbance')


fig.tight_layout()

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=7, sharex=True)
plot_nutrient_data_by_volume(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[0])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

plot_nutrient_data_by_volume(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[1])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')
rme_cleaned_uncorrected.loc['02-01-25':'11-01-25'].plot(y=['second_derivative'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:orange', secondary_y=True)


plot_nutrient_data_by_volume(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[2])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_volume(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[3])
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['Scaled PLSR'], ax=ax[3], rot=45, title='Corrected PLSR', c='tab:blue')

plot_nutrient_data_by_volume(rme_results, '02/01/25', '11/01/25', 'Nitrate', ax[4])
rme_cleaned_uncorrected.loc['02-01-25':'11-01-25'].plot(y=['Uncorrected Unscaled PLSR'], ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', c='tab:blue')

rme_cleaned_uncorrected.loc['02-01-25':'11-01-25'].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['220.0'], ax=ax[6], rot=45, label=['220nm Absorbance'], title='Corrected Absorbance')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['265.0'], ax=ax[6], rot=45, label=['265nm Absorbance'], title='Corrected Absorbance')


fig.tight_layout()

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=7, sharex=True)
plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[0])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[1])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[2])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[3])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['Scaled PLSR'], ax=ax[3], rot=45, title='Corrected PLSR', c='tab:blue')

plot_nutrient_data_by_type(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[4])
rme_cleaned_uncorrected.loc['03-15-2025':'05-01-2025'].plot(y=['Uncorrected Unscaled PLSR'], ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', c='tab:blue')

rme_cleaned_uncorrected.loc['03-15-2025':'05-01-2025'].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['220.0'], ax=ax[6], rot=45, label=['220nm Absorbance'], title='Corrected Absorbance')
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['265.0'], ax=ax[6], rot=45, label=['265nm Absorbance'], title='Corrected Absorbance')


fig.tight_layout()

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=5, sharex=True)
plot_nutrient_data_by_volume(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[0])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

plot_nutrient_data_by_volume(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[1])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_volume(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[2])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_volume(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[3])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['Scaled PLSR'], ax=ax[3], rot=45, title='Corrected PLSR', c='tab:blue')

plot_nutrient_data_by_volume(rme_results, '03-15-2025', '05-01-2025', 'Nitrate', ax[4])
rme_cleaned_uncorrected.loc['03-15-2025':'05-01-2025'].plot(y=['Uncorrected Unscaled PLSR'], ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', c='tab:blue')

fig.tight_layout()

# Correlation Plots

In [ ]:
rme_merged_clean = pd.merge_asof(rme_results, rme_cleaned, left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('1h')).dropna(subset=['Nitrate mean', 'two_wavelength_no3_mgl_correct'])
rme_merged_clean = rme_merged_clean.dropna(subset=['Nitrate mean', 'two_wavelength_no3_mgl_correct']) # drop two outliers from timeseries


In [ ]:
fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(11,11))

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl', kind='scatter', xerr='Nitrate err', ax=ax[0,0], title='Uncorrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl', .3, .6, .05, ax[0,0])

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[0,1], title='Corrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl_correct', .3, .8, .05, ax[0,1])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[1,0], title='Uncorrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl', .3, .15, .02, ax[1,0])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,1], title='Corrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl_correct', .3, .2, .03, ax[1,1])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[2,0], title='Uncorrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl', .3, .3, .03, ax[2,0])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[2,1], title='Corrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl_correct', .3, .3, .03, ax[2,1])

rme_merged_clean.plot(x='Nitrate mean', y='Scaled PLSR', xerr='Nitrate err',kind='scatter', ax=ax[3,1], title='Corrected PLSR')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'Scaled PLSR', .3, .3, .03, ax[3,1])

rme_merged_clean.plot(x='Nitrate mean', y='Uncorrected Unscaled PLSR', xerr='Nitrate err',kind='scatter', ax=ax[3,0], title='Uncorrected Unscaled PLSR')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'Uncorrected Unscaled PLSR', .3, .3, .03, ax[3,0])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl_bias_correct', xerr='Nitrate err',kind='scatter', ax=ax[2,2], title='Bias Corrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl_bias_correct', .3, .3, .03, ax[2,2])

fig.suptitle('RME Calibration Plots')
for ax_i in ax.flatten():
    if not ax_i.has_data():
        fig.delaxes(ax_i)
fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(11,11))

scatter0 = ax[0,0].scatter(rme_merged_clean['Nitrate mean'], rme_merged_clean['one_wavelength_no3_mgl'], c=rme_merged_clean['Water Volume'], cmap='bwr')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl', .3, .6, .05, ax[0,0])
ax[0,0].set_title('Uncorrected One Wavelength')
fig.colorbar(scatter0, ax=ax[0,0], label='Water Volume')

scatter1 = ax[0,1].scatter(rme_merged_clean['Nitrate mean'], rme_merged_clean['one_wavelength_no3_mgl_correct'], c=rme_merged_clean['Water Volume'], cmap='bwr')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl_correct', .3, .8, .05, ax[0,1])
ax[0,1].set_title('Corrected One Wavelength')
fig.colorbar(scatter1, ax=ax[0,1], label='Water Volume')

scatter2 = ax[1,0].scatter(rme_merged_clean['Nitrate mean'], rme_merged_clean['two_wavelength_no3_mgl'], c=rme_merged_clean['Water Volume'], cmap='bwr')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl', .3, .15, .02, ax[1,0])
ax[1,0].set_title('Uncorrected Two Wavelength')
fig.colorbar(scatter2, ax=ax[1,0], label='Water Volume')

scatter3 = ax[1,1].scatter(rme_merged_clean['Nitrate mean'], rme_merged_clean['two_wavelength_no3_mgl_correct'], c=rme_merged_clean['Water Volume'], cmap='bwr')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl_correct', .3, .1, .03, ax[1,1])
ax[1,1].set_title('Corrected Two Wavelength')
fig.colorbar(scatter3, ax=ax[1,1], label='Water Volume')

scatter4 = ax[2,0].scatter(rme_merged_clean['Nitrate mean'], rme_merged_clean['second_derivative_no3_mgl'], c=rme_merged_clean['Water Volume'], cmap='bwr')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl', .3, .3, .03, ax[2,0])
ax[2,0].set_title('Uncorrected Second Derivative')
fig.colorbar(scatter4, ax=ax[2,0], label='Water Volume')

scatter5 = ax[2,1].scatter(rme_merged_clean['Nitrate mean'], rme_merged_clean['second_derivative_no3_mgl_correct'], c=rme_merged_clean['Water Volume'], cmap='bwr')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl_correct', .3, .3, .03, ax[2,1])
ax[2,1].set_title('Corrected Second Derivative')
fig.colorbar(scatter5, ax=ax[2,1], label='Water Volume')

scatter6 = ax[3,1].scatter(rme_merged_clean['Nitrate mean'], rme_merged_clean['Scaled PLSR'], c=rme_merged_clean['Water Volume'], cmap='bwr')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'Scaled PLSR', .3, .3, .03, ax[3,1])
ax[3,1].set_title('Corrected PLSR')
fig.colorbar(scatter6, ax=ax[3,1], label='Water Volume')

scatter7 = ax[3,0].scatter(rme_merged_clean['Nitrate mean'], rme_merged_clean['Uncorrected Unscaled PLSR'], c=rme_merged_clean['Water Volume'], cmap='bwr')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'Uncorrected Unscaled PLSR', .3, .3, .03, ax[3,0])
ax[3,0].set_title('Uncorrected Unscaled PLSR')
fig.colorbar(scatter7, ax=ax[3,0], label='Water Volume')

scatter8 = ax[2,2].scatter(rme_merged_clean['Nitrate mean'], rme_merged_clean['second_derivative_no3_mgl_bias_correct'], c=rme_merged_clean['Water Volume'], cmap='bwr')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl_bias_correct', .3, .3, .03, ax[2,2])
ax[2,2].set_title('Bias Corrected Second Derivative')
fig.colorbar(scatter8, ax=ax[2,2], label='Water Volume')

fig.suptitle('RME Calibration Plots Colored by Water Volume')
for ax_i in ax.flatten():
    if not ax_i.has_data():
        fig.delaxes(ax_i)
fig.tight_layout()

In [ ]:
calibration_columns = ['one_wavelength_no3_mgl', 'one_wavelength_no3_mgl_correct', 'two_wavelength_no3_mgl', 'two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl', 'second_derivative_no3_mgl_correct', 'second_derivative_no3_mgl_bias_correct','Uncorrected Unscaled PLSR', 'Scaled PLSR']

for col in calibration_columns:
    rme_merged_clean[col + '_error']  = rme_merged_clean[col] - rme_merged_clean['Nitrate mean']

In [ ]:
fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(14, 16))

# Map columns to their positions in the grid
plot_grid = [
    ['one_wavelength_no3_mgl', 'one_wavelength_no3_mgl_correct', None],
    ['two_wavelength_no3_mgl', 'two_wavelength_no3_mgl_correct', None],
    ['second_derivative_no3_mgl', 'second_derivative_no3_mgl_correct', 'second_derivative_no3_mgl_bias_correct'],
    ['Uncorrected Unscaled PLSR', 'Scaled PLSR', None]
]

for row in range(4):
    for col in range(3):
        col_name = plot_grid[row][col]
        if col_name is not None:
            scatter = ax[row, col].scatter(
                rme_merged_clean['Water Volume'],
                rme_merged_clean[f'{col_name}_error'],
                c=rme_merged_clean['Nitrate mean'],
                cmap='bwr',
                alpha=0.7
            )
            ax[row, col].set_title(f'{col_name}')
            ax[row, col].set_xlabel('Water Volume')
            ax[row, col].set_ylabel('Error (Calibration - Nitrate mean)')
            ax[row, col].axhline(0, color='k', linestyle='--', linewidth=1)
            ax[row, col].axvline(200, color='r', linestyle='--', linewidth=1)
            fig.colorbar(scatter, ax=ax[row, col], label='Nitrate mean')
        else:
            fig.delaxes(ax[row, col])

fig.suptitle('Calibration Error vs. Water Volume (Colored by Nitrate mean)')
fig.tight_layout()


In [ ]:
fig, ax= plt.subplots(figsize=(8,4))

rme_results.plot(x='Water Volume', y='Nitrate mean', kind='scatter', alpha=0.7, ax=ax)
plt.xlabel('Water Volume')
plt.ylabel('Nitrate mean (mg/L)')
plt.title('Nitrate mean vs Water Volume')
plt.grid(True)
ax.axvline(200, color='r', linestyle='--')
ax.set_xlabel('Water Volume (mL)')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))

rme_merged_clean.plot(y='Water Volume', xlabel= 'Water Volume (mL)', kind='hist', bins= 25, ax=ax, cumulative=True, title='Cumulative Histogram of RME Sample Volumes')
ax.axvline(200, color='r', linestyle='--')

In [ ]:
rme_results.plot(y='Nitrate mean', kind='hist', xlabel = 'Nitrate Concentration (mg/L)')

# Plotting Without VOL Flags

In [ ]:
rme_results_vol
fig, ax= plt.subplots(figsize=(15,15), nrows=7, sharex=True)



rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')


rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['Scaled PLSR'], ax=ax[3], rot=45, title='Corrected PLSR', c='tab:blue')

rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['Uncorrected Unscaled PLSR'], ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', c='tab:blue')

rme_cleaned_uncorrected.loc['02-01-25':'11-01-25'].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')
rme_cleaned.loc['02-01-25':'11-01-25'].plot(y=['220.0', '265.0'], ax=ax[6], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title='Corrected Absorbance')

for i in range(0,5):
    plot_nutrient_data(rme_results_novol, '02/01/25', '11/01/25', 'Nitrate', ax[i], color='k')
    plot_nutrient_data(rme_results_vol, '02/01/25', '11/01/25', 'Nitrate', ax[i], color='r')
    handles, labels = ax[i].get_legend_handles_labels()
    labels[1] = 'Nitrate (No Vol)'
    labels[2] = 'Nitrate (Vol)'
    ax[i].legend(handles, labels)




fig.tight_layout()

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=7, sharex=True)
plot_nutrient_data_by_type(rme_results_novol, '03-15-2025', '05-01-2025', 'Nitrate', ax[0])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

plot_nutrient_data_by_type(rme_results_novol, '03-15-2025', '05-01-2025', 'Nitrate', ax[1])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_type(rme_results_novol, '03-15-2025', '05-01-2025', 'Nitrate', ax[2])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_type(rme_results_novol, '03-15-2025', '05-01-2025', 'Nitrate', ax[3])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['Scaled PLSR'], ax=ax[3], rot=45, title='Corrected PLSR', c='tab:blue')

plot_nutrient_data_by_type(rme_results_novol, '03-15-2025', '05-01-2025', 'Nitrate', ax[4])
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['Uncorrected Unscaled PLSR'], ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', c='tab:blue')

rme_cleaned_uncorrected.loc['03-15-2025':'05-01-2025'].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['220.0'], ax=ax[6], rot=45, label=['220nm Absorbance'], title='Corrected Absorbance')
rme_cleaned.loc['03-15-2025':'05-01-2025'].plot(y=['265.0'], ax=ax[6], rot=45, label=['265nm Absorbance'], title='Corrected Absorbance')


fig.tight_layout()

In [ ]:
fig, ax= plt.subplots(figsize=(15,15), nrows=7, sharex=True)
plot_nutrient_data_by_type(rme_results_novol, '02-15-2025', '06-01-2025', 'Nitrate', ax[0])
rme_cleaned.loc['02-15-2025':'06-01-2025'].plot(y=['two_wavelength_no3_mgl_correct'], ax=ax[0], rot=45, title='Corrected Two Wavelength', c='tab:blue')

plot_nutrient_data_by_type(rme_results_novol, '02-15-2025', '06-01-2025', 'Nitrate', ax[1])
rme_cleaned.loc['02-15-2025':'06-01-2025'].plot(y=['second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title='Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_type(rme_results_novol, '02-15-2025', '06-01-2025', 'Nitrate', ax[2])
rme_cleaned.loc['02-15-2025':'06-01-2025'].plot(y=['second_derivative_no3_mgl_bias_correct'], ax=ax[2], rot=45, title='Bias Corrected Second Derivative', c='tab:blue')


plot_nutrient_data_by_type(rme_results_novol, '02-15-2025', '06-01-2025', 'Nitrate', ax[3])
rme_cleaned.loc['02-15-2025':'06-01-2025'].plot(y=['Scaled PLSR'], ax=ax[3], rot=45, title='Corrected PLSR', c='tab:blue')

plot_nutrient_data_by_type(rme_results_novol, '02-15-2025', '06-01-2025', 'Nitrate', ax[4])
rme_cleaned.loc['02-15-2025':'06-01-2025'].plot(y=['Uncorrected Unscaled PLSR'], ax=ax[4], rot=45, title='Uncorrected Unscaled PLSR', c='tab:blue')

rme_cleaned_uncorrected.loc['02-15-2025':'06-01-2025'].plot(y=['220.0', '265.0'], ax=ax[5], rot=45, label=['220nm Absorbance', '265nm Absorbance'], title = 'Uncorrected Absorbance')
rme_cleaned.loc['02-15-2025':'06-01-2025'].plot(y=['220.0'], ax=ax[6], rot=45, label=['220nm Absorbance'], title='Corrected Absorbance')
rme_cleaned.loc['02-15-2025':'06-01-2025'].plot(y=['265.0'], ax=ax[6], rot=45, label=['265nm Absorbance'], title='Corrected Absorbance')


fig.tight_layout()

In [ ]:
rme_merged_clean_novol = pd.merge_asof(rme_results_novol, rme_cleaned, left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('1h')).dropna(subset=['Nitrate mean', 'two_wavelength_no3_mgl_correct'])


In [ ]:
rme_merged_clean_novol = rme_merged_clean_novol.drop('2025-02-24 03:30:00')

In [ ]:
fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(11,11))

rme_merged_clean_novol.plot(x='Nitrate mean', y='one_wavelength_no3_mgl', kind='scatter', xerr='Nitrate err', ax=ax[0,0], title='Uncorrected One Wavelength')
plot_linear_fit(rme_merged_clean_novol, rme_merged_clean_novol, 'Nitrate mean', 'one_wavelength_no3_mgl', .3, .6, .07, ax[0,0])

rme_merged_clean_novol.plot(x='Nitrate mean', y='one_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[0,1], title='Corrected One Wavelength')
plot_linear_fit(rme_merged_clean_novol, rme_merged_clean_novol, 'Nitrate mean', 'one_wavelength_no3_mgl_correct', .3, .8, .07, ax[0,1])

rme_merged_clean_novol.plot(x='Nitrate mean', y='two_wavelength_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[1,0], title='Uncorrected Two Wavelength')
plot_linear_fit(rme_merged_clean_novol, rme_merged_clean_novol, 'Nitrate mean', 'two_wavelength_no3_mgl', .3, .15, .07, ax[1,0])

rme_merged_clean_novol.plot(x='Nitrate mean', y='two_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,1], title='Corrected Two Wavelength')
plot_linear_fit(rme_merged_clean_novol, rme_merged_clean_novol, 'Nitrate mean', 'two_wavelength_no3_mgl_correct', .3, .2, .07, ax[1,1])

rme_merged_clean_novol.plot(x='Nitrate mean', y='second_derivative_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[2,0], title='Uncorrected Second Derivative')
plot_linear_fit(rme_merged_clean_novol, rme_merged_clean_novol, 'Nitrate mean', 'second_derivative_no3_mgl', .3, .3, .07, ax[2,0])

rme_merged_clean_novol.plot(x='Nitrate mean', y='second_derivative_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[2,1], title='Corrected Second Derivative')
plot_linear_fit(rme_merged_clean_novol, rme_merged_clean_novol, 'Nitrate mean', 'second_derivative_no3_mgl_correct', .3, .3, .07, ax[2,1])

rme_merged_clean_novol.plot(x='Nitrate mean', y='Scaled PLSR', xerr='Nitrate err',kind='scatter', ax=ax[3,1], title='Corrected PLSR')
plot_linear_fit(rme_merged_clean_novol, rme_merged_clean_novol, 'Nitrate mean', 'Scaled PLSR', .3, .25, .07, ax[3,1])

rme_merged_clean_novol.plot(x='Nitrate mean', y='Uncorrected Unscaled PLSR', xerr='Nitrate err',kind='scatter', ax=ax[3,0], title='Uncorrected Unscaled PLSR')
plot_linear_fit(rme_merged_clean_novol, rme_merged_clean_novol, 'Nitrate mean', 'Uncorrected Unscaled PLSR', .3, .25, .07, ax[3,0])

rme_merged_clean_novol.plot(x='Nitrate mean', y='second_derivative_no3_mgl_bias_correct', xerr='Nitrate err',kind='scatter', ax=ax[2,2], title='Bias Corrected Second Derivative')
plot_linear_fit(rme_merged_clean_novol, rme_merged_clean_novol, 'Nitrate mean', 'second_derivative_no3_mgl_bias_correct', .3, .25, .07, ax[2,2])

fig.suptitle('RME Calibration Plots')
for ax_i in ax.flatten():
    if not ax_i.has_data():
        fig.delaxes(ax_i)
fig.tight_layout()


In [ ]:
rme_merged_clean_novol.loc['07-01-25':, 'Nitrate mean'].describe()

# Plotting Other Nutrients

In [ ]:
rme_results_copy = rme_results
rme_results_copy['N-P Ratio mean'] = rme_results_copy['Nitrate mean']/rme_results_copy['Phosphate mean']
rme_results_copy['N-NH3 Ratio mean'] = rme_results_copy['Nitrate mean']/rme_results_copy['Ammonium mean']
rme_results_copy['N-P Ratio err'] = 0
rme_results_copy['N-NH3 Ratio err'] = 0


In [ ]:
fig, ax= plt.subplots(figsize=(15, 9),nrows=2,  sharex=True)
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'Nitrate', ax[0], color='green', title= 'Concentration')
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'Phosphate',  ax[0],color='red')
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'Ammonium', ax[0], color='blue' )

plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'N-P Ratio', ax[1], color='red', title= 'Ratio', logy=True)
plot_nutrient_data(rme_results_copy, '02/01/25', '11/01/25', 'N-NH3 Ratio', ax[1], color='blue', title= 'Ratio', logy=True)

ax[0].set_ylabel('Concentration (mg/L)')
ax[1].set_ylabel('Concentration Ratio')
ax[1].axhline(1, color='k', ls='--')


fig.tight_layout()

In [ ]:
fieldblank_results = pd.read_csv(result_dir+'field_blank_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)

# Export 



In [ ]:
rme_cleaned.to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv')
rme_cleaned_uncorrected.to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected.csv')
